<a href="https://colab.research.google.com/github/JUANOSORIOG/Senales_y_Sistemas/blob/main/DASHBOARD_Parcial_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Streamlit (interfaz web)
!pip install streamlit -q

# Procesamiento de sistemas
!pip install --upgrade control -q

# Audio y multimedia
!pip install soundfile yt-dlp -q
!pip install yt-dlp -q
!apt install ffmpeg -y > /dev/null

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 578.3/578.3 kB 13.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.3/174.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 71.4 MB/s eta 0:00:00




In [2]:
#instalar librerias necesarias para descargar audios youtube
!python3 -m pip install --force-reinstall https://github.com/yt-dlp/yt-dlp/archive/master.tar.gz -q
#Libreria para manipulacion de archivos de audio
!pip install soundfile -q
# Instalar o actualizar la librería control
!pip install --upgrade control

     / 2.8 MB 10.3 MB/s 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
!mkdir -p pages

In [4]:
%%writefile 0_Bienvenido.py
import streamlit as st

st.set_page_config(page_title="Parcial 2 - SyS 2025-I", layout="wide")

st.markdown("""

### Parcial2 SyS 2025-1


# Profesor: Andrés Marino Álvarez Meza


## Estudiante: Juan Esteban Osorio Gonzalez


**Este dashboard contiene las partes fundamentales que se deben de trabajar
a la hora de querer mostrar como se ven las señales segun el tipo de sistema que se quiera presentar

 **Parcial 2 de Señales y Sistemas 2025-I**.

---

### Módulo 1:

Modelado de sistemas:

- Mecánico (MRA) y Eléctrico (circuito RLC).

- Análisis de estabilidad.

- Respuestas temporales, diagrama de Bode, polos/ceros.

- Parámetros de desempeño del sistema.

### Módulo 2:

Modelo SSB-AM: Desarrollo matemático y visualización.

-Implementación con señales:.

-Audio (canción).

-Análisis completo: Dominio del tiempo y de la frecuencia.

- Modelado matemático y espectral.

- Señales: pulso rectangular y fragmento de canción real.

- Filtro IIR aplicado para recuperación.

- Visualización en tiempo y frecuencia."

---


""")

Writing 0_Bienvenido.py


In [5]:
%%writefile pages/1_Punto_1_MRD-RLC.py
import streamlit as st
import numpy as np
import matplotlib.pyplot as plt
from control import TransferFunction, bode, bode_plot, step_response
from control.matlab import step, impulse, pole
import pandas as pd

# Configuración general de la página en Streamlit
st.set_page_config(page_title="Punto 1 - MRD y RLC", layout="wide")
st.title(" Punto 1: Sistema Masa-Resorte-Amortiguador & Circuito RLC")

# ===========================
# ⚙️ Configuración en sidebar
# ===========================

# Encabezado del panel lateral
st.sidebar.header("Configuración del Sistema")

# Menú desplegable para seleccionar el tipo de respuesta del sistema
tipo_respuesta = st.sidebar.selectbox(
    "Tipo de Respuesta",
    ["Subamortiguada", "Sobreamortiguada", "Amortiguamiento Crítico", "Inestable"]
)

# Deslizadores para seleccionar frecuencia natural y factor de amortiguamiento
wn = st.sidebar.slider("Frecuencia Natural (ωₙ)", 0.1, 20.0, 5.0)
zeta = st.sidebar.slider("Factor de Amortiguamiento (ζ)", 0.0, 2.0, 0.5)

# Ajustar el valor de ζ dependiendo del tipo de respuesta seleccionada
if tipo_respuesta == "Subamortiguada":
    zeta = st.sidebar.slider("ζ", 0.0, 1.0, 0.3)
elif tipo_respuesta == "Sobreamortiguada":
    zeta = st.sidebar.slider("ζ", 1.0, 2.0, 1.5)
elif tipo_respuesta == "Amortiguamiento Crítico":
    zeta = 1.0
elif tipo_respuesta == "Inestable":
    zeta = st.sidebar.slider("ζ", -1.0, 0.0, -0.1)

# ====================================
# 🛠️ Definición del sistema mecánico
# ====================================

# Función de transferencia: H(s) = 1 / (ms² + bs + k)
m = 1.0                              # Masa
k = wn**2                            # Constante del resorte
b = 2 * zeta * wn * m                # Coeficiente de amortiguamiento
num_mec = [1]                        # Numerador de H(s)
den_mec = [m, b, k]                  # Denominador de H(s)
sys_mec = TransferFunction(num_mec, den_mec)  # Sistema mecánico como TF

# ============================================
# 🔌 Sistema eléctrico equivalente (RLC serie)
# ============================================

# Equivalencias: L = m, C = 1/k, R = b
L = m
C = m / L
R = L / b
num_elec = [R*C, 0]                 # Numerador de la TF del circuito
den_elec = [L*C, R*C, 1]           # Denominador
sys_elec = TransferFunction(num_elec, den_elec)  # Sistema eléctrico como TF

# ==========================
# 📋 Mostrar parámetros físicos
# ==========================

col1, col2 = st.columns(2)

# Parámetros del sistema mecánico
col1.subheader("🔧 Sistema Mecánico")
col1.write(f"Masa (m): {m:.2f} kg")
col1.write(f"Amortiguador (b): {b:.2f} Ns/m")
col1.write(f"Resorte (k): {k:.2f} N/m")

# Parámetros del sistema eléctrico equivalente
col2.subheader("🔌 Sistema Eléctrico")
col2.write(f"Inductor (L): {L:.2f} H")
col2.write(f"Capacitor (C): {C:.4f} F")
col2.write(f"Resistor (R): {R:.2f} Ω")

# ============================
# 📈 Funciones para graficar
# ============================

# Diagrama de Bode
def plot_bode(sys, title):
    w = np.logspace(-2, 2, 1000)  # Rango logarítmico de frecuencias
    fig, ax = plt.subplots(2, 1, figsize=(10, 6))  # Crear figura con dos subgráficas
    try:
        bode_plot(sys, omega=w, dB=True, Hz=False, deg=True)  # Graficar automáticamente
        plt.suptitle(title)  # Título general
        st.pyplot(fig)  # Mostrar en Streamlit
    except Exception as e:
        st.error(f"⚠ Error al graficar diagrama de Bode: {e}")

# Respuesta al escalón
def plot_step(sys, title):
    try:
        t, y = step_response(sys)  # Obtener tiempo y respuesta
        plt.figure(figsize=(10, 4))
        plt.plot(t, y)             # Graficar
        plt.grid()
        plt.xlabel('Tiempo (s)')
        plt.ylabel('Respuesta')
        plt.title(title)
        st.pyplot(plt)
    except Exception as e:
        st.error(f"⚠ Error al graficar respuesta al escalón: {e}")

# Mapa de polos (no se grafican ceros)
def plot_pzmap(sys, title):
    p = pole(sys)  # Obtener polos del sistema
    z = []         # No se calculan ceros explícitamente
    plt.figure(figsize=(6, 6))
    plt.scatter(np.real(p), np.imag(p), marker='x', color='red', label='Polos')
    plt.axhline(0, color='black', lw=1)  # Eje real
    plt.axvline(0, color='black', lw=1)  # Eje imaginario
    plt.grid(True)
    plt.legend()
    plt.title(title)
    st.pyplot(plt)

# =======================================
# ⏱️ Cálculo de parámetros temporales
# Solo si el sistema es subamortiguado
# =======================================
if tipo_respuesta == "Subamortiguada" and 0 < zeta < 1:
    try:
        wd = wn * np.sqrt(1 - zeta**2)  # Frecuencia angular amortiguada
        tr = np.pi / wd                 # Tiempo de levantamiento
        tp = np.pi / wd                 # Tiempo al pico
        Mp = np.exp(-zeta * np.pi / np.sqrt(1 - zeta**2))  # Sobreimpulso máximo
        ts = 4 / (zeta * wn)            # Tiempo de establecimiento (2%)

        # Mostrar resultados en tabla
        st.markdown("##  Parámetros Temporales (Subamortiguado) 📊 ")
        params_df = pd.DataFrame({
            "Parámetro": [
                "Tiempo de levantamiento (tr)",
                "Sobreimpulso máximo (Mp)",
                "Tiempo al pico (tp)",
                "Tiempo de establecimiento (ts)"
            ],
            "Valor": [
                f"{tr:.3f} s",
                f"{Mp*100:.2f}%",
                f"{tp:.3f} s",
                f"{ts:.3f} s"
            ]
        })
        st.table(params_df)
    except Exception as e:
        st.warning("⚠ No se pudieron calcular los parámetros dinámicos.")

# =======================================
# 🖼️ Layout de visualización de gráficas
# =======================================
st.markdown("## Análisis del Sistema 🔍")
col1, col2 = st.columns(2)

# Panel izquierdo: sistema mecánico
with col1:
    st.markdown("###  Sistema Mecánico 🧑‍🔧")
    plot_bode(sys_mec, "Bode - Sistema Mecánico")
    plot_pzmap(sys_mec, "Polos y Ceros - Sistema Mecánico")
    plot_step(sys_mec, "Respuesta al Escalón - Sistema Mecánico")

# Panel derecho: sistema eléctrico equivalente
with col2:
    st.markdown("###  Sistema Eléctrico ⚡")
    plot_bode(sys_elec, "Bode - Sistema Eléctrico")
    plot_pzmap(sys_elec, "Polos y Ceros - Sistema Eléctrico")
    plot_step(sys_elec, "Respuesta al Escalón - Sistema Eléctrico")

Writing pages/1_Punto_1_MRD-RLC.py


In [6]:
%%writefile pages/3_Punto2_SSB_AM_Interactivo.py
# 3_SSB_AM_Interactivo.py
import streamlit as st
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
from scipy.signal import (
    hilbert, butter, cheby1, cheby2, bessel, ellip,
    freqz, tf2zpk, lfilter
)
from pydub import AudioSegment
import yt_dlp
import os
from io import BytesIO

st.set_page_config(page_title="🎙 SSB-AM Dashboard", layout="wide")
st.title("🎙 Explorador Interactivo de Modulación SSB-AM")

def array_to_audiosegment(signal, sample_rate):
    signal_int16 = np.int16(signal / np.max(np.abs(signal)) * 32767)
    audio = AudioSegment(
        signal_int16.tobytes(),
        frame_rate=sample_rate,
        sample_width=2,
        channels=1
    )
    return audio

def export_audiosegment_to_bytes(audiosegment):
    buf = BytesIO()
    audiosegment.export(buf, format="wav")
    return buf.getvalue()

option = st.radio("Selecciona el tipo de señal mensaje:", ["Canción de YouTube", "Pulso rectangular"])
proceso_exitoso = False

if option == "Canción de YouTube":
    url = st.text_input("🎧 Ingresa el enlace de YouTube:", "")
    if url:
        try:
            with st.spinner("🔍 Descargando y procesando audio..."):
                ydl_opts = {
                    'format': 'bestaudio/best',
                    'outtmpl': 'audio.%(ext)s',
                    'postprocessors': [{
                        'key': 'FFmpegExtractAudio',
                        'preferredcodec': 'mp3',
                    }],
                    'quiet': True,
                    'no_warnings': True,
                }
                with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                    ydl.download([url])
                audio = AudioSegment.from_file("audio.mp3").set_channels(1).set_frame_rate(44100)
                fragment = audio[20000:25000]
                os.remove("audio.mp3")
                samples = np.array(fragment.get_array_of_samples()).astype(np.float32) / 2**15
                Fs = fragment.frame_rate
                t = np.linspace(0, len(samples)/Fs, len(samples))
                m_t = samples
                proceso_exitoso = True
        except Exception as e:
            st.error(f"❌ Error al procesar audio: {e}")
            st.stop()
elif option == "Pulso rectangular":
    Fs = 44100
    duration = 1.0
    t = np.linspace(0, duration, int(Fs * duration), endpoint=False)
    m_t = np.where((t > 0.3) & (t < 0.7), 1.0, 0.0)
    proceso_exitoso = True

if proceso_exitoso:
    filtro_tipo = st.selectbox("🔧 Tipo de filtro IIR para demodulación:", [
        "Butterworth", "Chebyshev I", "Chebyshev II", "Bessel", "Elíptico"
    ])
    orden = st.slider("🔢 Orden del filtro:", 2, 10, 4)
    Fc_corte = st.slider("🔽 Frecuencia de corte [Hz]:", 1000, 20000, 8000)

    Fc = 10000
    analytic_signal = hilbert(m_t)
    ssb = np.real(analytic_signal * np.exp(1j * 2 * np.pi * Fc * t))
    portadora = np.cos(2 * np.pi * Fc * t)
    mixed = ssb * portadora

    Wn = Fc_corte / (Fs / 2)
    rp, rs = 1, 40
    if filtro_tipo == "Butterworth":
        b, a = butter(orden, Wn, btype='low')
    elif filtro_tipo == "Chebyshev I":
        b, a = cheby1(orden, rp, Wn, btype='low')
    elif filtro_tipo == "Chebyshev II":
        b, a = cheby2(orden, rs, Wn, btype='low')
    elif filtro_tipo == "Bessel":
        b, a = bessel(orden, Wn, btype='low', norm='phase')
    elif filtro_tipo == "Elíptico":
        b, a = ellip(orden, rp, rs, Wn, btype='low')

    demod = lfilter(b, a, mixed)

    if option == "Canción de YouTube":
        audio_m = fragment
        audio_modulada = array_to_audiosegment(ssb, Fs)
        audio_demodulada = array_to_audiosegment(demod, Fs)

        st.subheader("Escucha las señales")
        st.audio(export_audiosegment_to_bytes(audio_m), format="audio/wav")
        st.caption("Señal Mensaje")
        st.audio(export_audiosegment_to_bytes(audio_modulada), format="audio/wav")
        st.caption("Señal Modulada SSB")
        st.audio(export_audiosegment_to_bytes(audio_demodulada), format="audio/wav")
        st.caption("Señal Demodulada")

    st.subheader("Señales en el dominio del tiempo")
    fig, axs = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
    axs[0].plot(t, m_t, color='purple'); axs[0].set_title("Señal Mensaje m(t)"); axs[0].grid()
    axs[1].plot(t, ssb, color='blue'); axs[1].set_title("Señal Modulada SSB"); axs[1].grid()
    axs[2].plot(t, mixed, color='red'); axs[2].set_title("Señal Mezclada (SSB * cos)"); axs[2].grid()
    axs[3].plot(t, demod, color='orange'); axs[3].set_title("Señal Demodulada"); axs[3].set_xlabel("Tiempo (s)"); axs[3].grid()
    st.pyplot(fig)

    st.subheader("Espectros de Frecuencia")
    def plot_spectrum(signal, Fs, title, color):
        N = len(signal)
        spectrum = fft(signal)
        freqs = fftfreq(N, 1/Fs)
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(freqs[:N//2], np.abs(spectrum[:N//2]), color=color)
        ax.set_title(title)
        ax.set_xlabel("Frecuencia (Hz)")
        ax.set_ylabel("Magnitud")
        ax.grid()
        return fig

    st.pyplot(plot_spectrum(m_t, Fs, "Espectro de la Señal Mensaje", 'purple'))
    st.pyplot(plot_spectrum(ssb, Fs, "Espectro de la Señal Modulada SSB", 'blue'))
    st.pyplot(plot_spectrum(mixed, Fs, "Espectro de la Señal Mezclada", 'red'))
    st.pyplot(plot_spectrum(demod, Fs, "Espectro de la Señal Demodulada", 'orange'))

    st.subheader("📉 Diagrama de Bode del Filtro")
    w, h = freqz(b, a, worN=8000)
    fig_bode, ax = plt.subplots()
    ax.plot(w * Fs / (2 * np.pi), 20 * np.log10(abs(h)))
    ax.set_title(f'Bode - {filtro_tipo}')
    ax.set_xlabel('Frecuencia [Hz]')
    ax.set_ylabel('Ganancia [dB]')
    ax.grid()
    st.pyplot(fig_bode)

    st.subheader("Plano de Polos y Ceros del Filtro")
    z, p, _ = tf2zpk(b, a)
    fig_pz, ax = plt.subplots()
    theta = np.linspace(0, 2*np.pi, 300)
    ax.plot(np.cos(theta), np.sin(theta), 'k--', linewidth=1)
    ax.scatter(np.real(z), np.imag(z), marker='o', color='blue', label='Ceros')
    ax.scatter(np.real(p), np.imag(p), marker='x', color='red', label='Polos')
    for i, pole in enumerate(p):
        location = "Izquierda" if np.real(pole) < 0 else "Derecha"
        ax.annotate(f"P{i+1} ({location})", (np.real(pole), np.imag(pole)),
                    textcoords="offset points", xytext=(5,5), fontsize=8)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_xlim([-1.5, 1.5])
    ax.set_ylim([-1.5, 1.5])
    ax.set_title(f"Polos y Ceros - {filtro_tipo}")
    ax.set_xlabel("Re{z}")
    ax.set_ylabel("Im{z}")
    ax.grid()
    ax.legend()
    ax.set_aspect('equal')
    st.pyplot(fig_pz)

    st.success("✅ Análisis completado. Puedes cambiar el filtro o la señal para continuar explorando.")

Writing pages/3_Punto2_SSB_AM_Interactivo.py


In [7]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!mv cloudflared-linux-amd64 /usr/local/bin/cloudflared

#Ejecutar Streamlit
!streamlit run 0_Bienvenido.py &>/content/logs.txt & #Cambiar 0_👋_Hello.py por el nombre de tu archivo principal

#Exponer el puerto 8501 con Cloudflare Tunnel
!cloudflared tunnel --url http://localhost:8501 > /content/cloudflared.log 2>&1 &

#Leer la URL pública generada por Cloudflare
import time
time.sleep(5)  # Esperar que se genere la URL

import re
found_context = False  # Indicador para saber si estamos en la sección correcta

with open('/content/cloudflared.log') as f:
    for line in f:
        #Detecta el inicio del contexto que nos interesa
        if "Your quick Tunnel has been created" in line:
            found_context = True

        #Busca una URL si ya se encontró el contexto relevante
        if found_context:
            match = re.search(r'https?://\S+', line)
            if match:
                url = match.group(0)  #Extrae la URL encontrada
                print(f'Tu aplicación está disponible en: {url}')
                break  #Termina el bucle después de encontrar la URL

--2025-07-17 22:19:03--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.116.3
Connecting to github.com (github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2025.7.0/cloudflared-linux-amd64 [following]
--2025-07-17 22:19:03--  https://github.com/cloudflare/cloudflared/releases/download/2025.7.0/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/37d2bad8-a2ed-4b93-8139-cbb15162d81d?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-07-17T23%3A01%3A20Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-07-17T2